In [38]:
import pandas as pd
import librosa
import numpy as np
import os
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score


In [39]:
df = pd.read_csv('../data/labels/bnd_voices.csv')
df.head()

,song,start,finish,member
0,dear_my_darling,0:08,0:22,Sungho
1,dear_my_darling,0:23,0:36,Riwoo
2,dear_my_darling,0:37,0:51,Jaehyun
3,dear_my_darling,0:52,1:07,Woonhak
4,dear_my_darling,1:08,1:20,Leehan


In [40]:
df.dtypes

song      str
start     str
finish    str
member    str
dtype: object

In [41]:
def convert_to_seconds(time_str):
    parts = time_str.split(":")
    minutes = int(parts[0])
    seconds = int(parts[1])
    total = minutes * 60 + seconds
    return total

convert_to_seconds("1:20")

80

In [42]:
df.columns = df.columns.str.strip()
df.columns
df["start_sec"] = df["start"].apply(convert_to_seconds)
df["finish_sec"] = df["finish"].apply(convert_to_seconds)
df.head()

,song,start,finish,member,start_sec,finish_sec
0,dear_my_darling,0:08,0:22,Sungho,8,22
1,dear_my_darling,0:23,0:36,Riwoo,23,36
2,dear_my_darling,0:37,0:51,Jaehyun,37,51
3,dear_my_darling,0:52,1:07,Woonhak,52,67
4,dear_my_darling,1:08,1:20,Leehan,68,80


In [43]:
import soundfile as sf



In [44]:
for index, row in df.iterrows():
    print(row["song"], row["start_sec"], row["finish_sec"], row["member"])

dear_my_darling 8 22 Sungho
dear_my_darling 23 36 Riwoo
dear_my_darling 37 51 Jaehyun
dear_my_darling 52 67 Woonhak
dear_my_darling 68 80 Leehan
dear_my_darling 81 97 Taesan
123_78_ 14 20 Jaehyun
123_78_ 21 27 Leehan
123_78_ 28 37 Sungho
123_78_ 38 42 Riwoo
123_78_ 43 44 Leehan
123_78_ 45 51 Woonhak
123_78_ 52 58 Taesan
123_78_ 59 70 Riwoo
123_78_ 71 79 Woonhak
123_78_ 80 83 Jaehyun
123_78_ 84 93 Sungho
123_78_ 94 97 Leehan
123_78_ 98 100 Jaehyun
123_78_ 101 107 Riwoo
123_78_ 108 114 Taesan
123_78_ 115 125 Jaehyun
123_78_ 143 149 Leehan
123_78_ 150 155 Sungho
reality 21 184 Taesan
iktoy 13 140 Taesan
smart 8 73 Jaehyun
sway 15 108 Leehan
ttte 8 60 Sungho
winner 0 83 Woonhak
darling 8 74 Riwoo


itterrows used


In [45]:
unique_songs = df["song"].unique()

for song_name in unique_songs:
    y,sr = librosa.load(f"../data/raw_audio/{song_name}.mp3", sr=None)
    song_rows = df[df["song"]==song_name]
    for index, row in song_rows.iterrows():
        start_sample = int(row["start_sec"] * sr)
        end_sample = int(row["finish_sec"] * sr)
        clip = y[start_sample:end_sample]
        sf.write(f"../data/processed/{row['member']}_{index}.wav", clip, sr)



In [46]:
def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=None)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std = mfcc.std(axis=1)
    feature_vector = np.concatenate([mfcc_mean, mfcc_std])
    return feature_vector

test_features = extract_features("../data/processed/Sungho_0.wav")
print(test_features.shape)

(26,)


In [47]:
folder_path = "../data/processed/"
all_files = os.listdir(folder_path)
print(all_files)


['Jaehyun_15.wav', 'Jaehyun_18.wav', 'Jaehyun_2.wav', 'Jaehyun_21.wav', 'Jaehyun_26.wav', 'Jaehyun_6.wav', 'Leehan_10.wav', 'Leehan_17.wav', 'Leehan_22.wav', 'Leehan_27.wav', 'Leehan_4.wav', 'Leehan_7.wav', 'Riwoo_1.wav', 'Riwoo_13.wav', 'Riwoo_19.wav', 'Riwoo_30.wav', 'Riwoo_9.wav', 'Sungho_0.wav', 'Sungho_16.wav', 'Sungho_23.wav', 'Sungho_28.wav', 'Sungho_8.wav', 'Taesan_12.wav', 'Taesan_20.wav', 'Taesan_24.wav', 'Taesan_25.wav', 'Taesan_5.wav', 'test', 'train', 'Woonhak_11.wav', 'Woonhak_14.wav', 'Woonhak_29.wav', 'Woonhak_3.wav']


Test faylinin organize edilmesi

In [48]:
df_test = pd.read_csv("../data/labels/bnd_voices_test.csv")
df_test.columns = df_test.columns.str.strip()
df_test.head()

,song,start,finish,member
0,dangerous,0:08,0:17,Taesan
1,dangerous,0:18,0:25,Leehan
2,dangerous,0:26,0:33,Woonhak
3,dangerous,0:34,0:42,Jaehyun
4,dangerous,0:44,0:51,Sungho


In [49]:
df_test["start_sec"] = df_test["start"].apply(convert_to_seconds)
df_test["finish_sec"] = df_test["finish"].apply(convert_to_seconds)
df_test.head()

,song,start,finish,member,start_sec,finish_sec
0,dangerous,0:08,0:17,Taesan,8,17
1,dangerous,0:18,0:25,Leehan,18,25
2,dangerous,0:26,0:33,Woonhak,26,33
3,dangerous,0:34,0:42,Jaehyun,34,42
4,dangerous,0:44,0:51,Sungho,44,51


In [50]:
y, sr = librosa.load("../data/raw_audio/dangerous.mp3", sr=None)

for index, row in df_test.iterrows():
    start_sample = int(row["start_sec"] * sr)
    end_sample = int(row["finish_sec"] * sr)
    clip = y[start_sample:end_sample]
    sf.write(f"../data/processed/test/{row['member']}_{index}.wav", clip, sr)

Train


In [51]:
train_folder = "../data/processed/train/"
train_files = os.listdir(train_folder)

X_train_list = []
y_train_list = []

for filename in train_files:
    member_name = filename.split("_")[0]
    y_train_list.append(member_name)
    
    file_path = train_folder + filename
    features = extract_features(file_path)
    X_train_list.append(features)

X_train = np.array(X_train_list)
y_train = np.array(y_train_list)

print(X_train.shape)
print(y_train.shape)

(31, 26)
(31,)


In [52]:
print(set(y_train))
from collections import Counter
print(Counter(y_train))

{np.str_('Riwoo'), np.str_('Leehan'), np.str_('Sungho'), np.str_('Taesan'), np.str_('Woonhak'), np.str_('Jaehyun')}
Counter({np.str_('Jaehyun'): 6, np.str_('Leehan'): 6, np.str_('Riwoo'): 5, np.str_('Sungho'): 5, np.str_('Taesan'): 5, np.str_('Woonhak'): 4})


Test


In [53]:
test_folder = "../data/processed/test/"
test_files = os.listdir(test_folder)

X_test_list = []
y_test_list = []

for filename in test_files:
    member_name = filename.split("_")[0]
    y_test_list.append(member_name)
    
    file_path = test_folder + filename
    features = extract_features(file_path)
    X_test_list.append(features)

X_test = np.array(X_test_list)
y_test = np.array(y_test_list)

print(X_test.shape)
print(y_test.shape)
print(set(y_test))

(14, 26)
(14,)
{np.str_('Riwoo'), np.str_('Leehan'), np.str_('Sungho'), np.str_('Taesan'), np.str_('Woonhak'), np.str_('Jaehyun')}


In [54]:
model = KNeighborsClassifier(n_neighbors=8)
model.fit(X_train, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",8
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[<U7](6,)","['Jaehyun','Leehan','Riwoo','Sungho','Taesan','Woonhak']"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


changed k 3 to 8 bcs of distance between test and training data error

In [55]:
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")

Accuracy: 28.57%


In [56]:
print(confusion_matrix(y_test, y_pred, labels=list(set(y_test))))
print(list(set(y_test)))

[[2 0 0 0 0 0]
 [0 0 0 0 0 2]
 [0 0 0 0 0 2]
 [0 0 0 0 0 3]
 [0 0 0 0 0 2]
 [1 0 0 0 0 2]]
[np.str_('Riwoo'), np.str_('Leehan'), np.str_('Sungho'), np.str_('Taesan'), np.str_('Woonhak'), np.str_('Jaehyun')]


In [57]:
print(X_train)

[[-7.12893982e+01  1.38887894e+02 -4.20865250e+01  6.05884933e+01
  -2.15031300e+01  3.59917717e+01 -1.48545790e+01  1.23755646e+01
  -8.86124325e+00  3.96411896e+00 -1.92983007e+00 -1.90751183e+00
   4.67735815e+00  5.38991966e+01  3.33425980e+01  1.86704006e+01
   2.08214951e+01  1.95451527e+01  1.31402588e+01  1.32126236e+01
   9.28304672e+00  1.25759850e+01  9.74850273e+00  9.10390472e+00
   8.01821041e+00  7.95010138e+00]
 [-1.02488045e+02  1.33703369e+02 -4.05562325e+01  5.82842751e+01
  -1.30726662e+01  3.84795265e+01 -8.09820271e+00  1.63662434e+01
  -3.86690521e+00  5.63466978e+00  5.40942669e+00  5.06215191e+00
   9.03415966e+00  5.84836693e+01  2.62224712e+01  2.54620876e+01
   1.61099072e+01  2.54506073e+01  1.98607540e+01  1.69729977e+01
   1.02883997e+01  1.06092472e+01  1.08205328e+01  9.58785057e+00
   9.07210922e+00  8.04514885e+00]
 [-8.64342575e+01  1.58733551e+02 -2.23159523e+01  3.64354591e+01
  -3.08516884e+01  2.63069668e+01 -8.43927860e+00  8.53743649e+00
  -1.0

Comparing
train_accuracies = []
test_accuracies = []
k_values = range(1, 11)

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    train_acc = accuracy_score(y_train, y_train_pred)
    train_accuracies.append(train_acc)
    
    y_test_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_accuracies.append(test_acc)

print("Train:", train_accuracies)
print("Test:", test_accuracies)

import matplotlib.pyplot as plt

plt.plot(k_values, train_accuracies, marker='o', label='Train Accuracy')
plt.plot(k_values, test_accuracies, marker='o', label='Test Accuracy')
plt.xlabel("n_neighbors (k)")
plt.ylabel("Accuracy")
plt.title("kNN: Train vs Test Accuracy")
plt.legend()
plt.grid(True)
plt.show()